# RuleSum (Colab Demo) — Best Structured Method vs Zero-shot

This Colab notebook lets you:
- Paste a legal document (medium length).
- Visualize step-by-step how the **Best structured method** works:
  1. **KG (IRAC-aware)** extraction  
  2. **IRAC-Quota Top-K selection** of triples  
  3. **Kaping-IRAC serialization** of triples  
  4. **IRAC-guided summary (~150 words)**
- Compare with a **Zero-shot baseline** summary (no KG, no IRAC).

It also renders a **Plain KG** (no IRAC constraints) so you can see the difference.


In [ ]:
!pip -q install \
  gradio>=4.38.1 \
  networkx>=3.2 \
  pydot>=2.0.0 \
  graphviz>=0.20.3

!apt-get -qq update && apt-get -qq install -y graphviz > /dev/null


In [ ]:
!pip -q install -U "langchain-openai>=0.1.24" "langchain>=0.2.15" \
                 "langchain-community>=0.2.12" "langchain-experimental>=0.0.64" \
                 "openai>=1.43.0" "httpx>=0.27.0" \
                 "datasets==3.6.0" "rouge-score" "sentence-transformers>=3.0.1" \
                 "textstat" "orjson"


In [ ]:
import os
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    # Paste your key here or set it in the environment
    OPENAI_API_KEY = ""
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))


In [ ]:
from datasets import load_dataset
ds = load_dataset("allenai/multi_lexsum", name="v20230518")

In [ ]:
import os, re, io, base64, tempfile, textwrap, glob
from dataclasses import dataclass
from typing import List, Optional, Dict

import numpy as np
import networkx as nx
import pandas as pd
import textstat as _ts

import httpx
from openai import OpenAI
from sentence_transformers import SentenceTransformer

from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
try:
    from langchain_core.documents import Document
except:
    from langchain.schema import Document

# ---- Tunables ----
OPENAI_CHAT_MODEL = os.getenv("RULESUM_MODEL", "gpt-4o-mini")  # set RULESUM_MODEL="gpt-4o" for best quality
TEMPERATURE = float(os.getenv("RULESUM_TEMPERATURE", "0.2"))
MAX_DOC_CHARS = int(os.getenv("RULESUM_MAX_DOC_CHARS", "8000"))
TOPK_TRIPLES = int(os.getenv("RULESUM_TOPK", "30"))
LENGTH_WORDS = int(os.getenv("RULESUM_LEN", "150"))  # "medium"

client = OpenAI()
llm = ChatOpenAI(model=OPENAI_CHAT_MODEL, temperature=TEMPERATURE, http_client=httpx.Client())
_embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


In [ ]:
sample_text = ds["train"]["sources"][0]

# Save it to a local file
with open("sample_case_0.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(sample_text))

print("Saved first training source to sample_case_0.txt")

In [ ]:
@dataclass
class Triple:
    s: str; r: str; o: str

def _norm(x: str) -> str:
    return re.sub(r"\s+"," ", (x or "").strip())


In [ ]:
def build_plain_kg(text: str) -> List[Triple]:
    t = LLMGraphTransformer(llm=llm, strict_mode=False)
    gdocs = t.convert_to_graph_documents([Document(page_content=text)])
    triples=[]
    for gd in gdocs:
        for rel in getattr(gd, "relationships", []):
            s = getattr(rel.source, "id", str(rel.source))
            o = getattr(rel.target, "id", str(rel.target))
            r = getattr(rel, "type", "")
            if s and r and o:
                triples.append(Triple(_norm(s), _norm(r), _norm(o)))
    uniq, seen = [], set()
    for tr in triples:
        key = (tr.s.lower(), tr.r.lower(), tr.o.lower())
        if key not in seen:
            seen.add(key); uniq.append(tr)
    return uniq

def build_irac_kg(text: str) -> List[Triple]:
    ALLOWED_NODES = ["Case","Issue","Rule","Application","Conclusion","Court","Party","Fact",
                     "Precedent","Statute","Jurisdiction","Date","Outcome","Remedy"]
    ALLOWED_REL = [
        ("Case","HAS_ISSUE","Issue"), ("Issue","ADDRESSED_BY","Rule"),
        ("Rule","APPLIED_IN","Application"), ("Application","SUPPORTS","Conclusion"),
        ("Case","HAS_CONCLUSION","Conclusion"), ("Case","CITES","Precedent"),
        ("Rule","DERIVED_FROM","Precedent"), ("Rule","DERIVED_FROM","Statute"),
        ("Case","DECIDED_BY","Court"), ("Case","IN_JURISDICTION","Jurisdiction"),
        ("Case","HAS_DATE","Date"), ("Case","INVOLVES","Party"),
        ("Application","USES_FACT","Fact"),
    ]
    from langchain_core.prompts import ChatPromptTemplate
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract a knowledge graph using IRAC; normalize labels to this schema."),
        ("human", "{input}")
    ])
    t = LLMGraphTransformer(llm=llm, allowed_nodes=ALLOWED_NODES,
                            allowed_relationships=ALLOWED_REL, strict_mode=True, prompt=prompt)
    gdocs = t.convert_to_graph_documents([Document(page_content=text)])

    def _canon(r: str)->str:
        m = {"APPLIES":"APPLIED_IN","APPLIES_TO":"APPLIED_IN","ANALYZES_UNDER":"APPLIED_IN",
             "ANALYZES":"APPLIED_IN","LEADS_TO":"SUPPORTS","WARRANTS":"SUPPORTS",
             "REFERS_TO":"CITES","RELIES_ON":"CITES","REFERENCES":"CITES",
             "BASED_ON":"DERIVED_FROM","GOVERNED_BY":"ADDRESSED_BY","ANSWERED_BY":"ADDRESSED_BY",
             "RESOLVED_BY":"ADDRESSED_BY"}
        return m.get((r or "").strip().upper(), r)

    triples=[]
    for gd in gdocs:
        for rel in getattr(gd, "relationships", []):
            s = getattr(rel.source, "id", str(rel.source))
            o = getattr(rel.target, "id", str(rel.target))
            r = _canon(getattr(rel, "type", ""))
            if s and r and o:
                triples.append(Triple(_norm(s), _norm(r), _norm(o)))
    uniq, seen = [], set()
    for tr in triples:
        key = (tr.s.lower(), tr.r.lower(), tr.o.lower())
        if key not in seen:
            seen.add(key); uniq.append(tr)
    return uniq


In [ ]:
def _irac_bucket_of(rel: str) -> str:
    m = {"HAS_ISSUE":"ISSUE","ADDRESSED_BY":"RULE","APPLIED_IN":"APPLICATION",
         "SUPPORTS":"CONCLUSION","HAS_CONCLUSION":"CONCLUSION","CITES":"RULE",
         "DERIVED_FROM":"RULE","USES_FACT":"APPLICATION"}
    return m.get((rel or "").strip().upper(), "FACTS")

def select_topk_with_irac_quota(triples: List[Triple], doc_text: str, k: int=30,
                                quota: Optional[Dict[str,int]]=None) -> List[Triple]:
    from collections import defaultdict
    if not triples or k<=0: return []
    quota = quota or {"ISSUE":1, "RULE":1, "APPLICATION":1, "CONCLUSION":1}
    by_bucket = defaultdict(list)
    for t in triples: by_bucket[_irac_bucket_of(t.r)].append(t)

    doc_emb = _embedder.encode([doc_text], normalize_embeddings=True)
    picked=[]
    # guarantee coverage per section
    for sec, q in quota.items():
        cands = by_bucket.get(sec, [])
        if not cands or q<=0: continue
        texts = [f"{t.s} {t.r} {t.o}" for t in cands]
        emb = _embedder.encode(texts, normalize_embeddings=True)
        sims = (emb @ doc_emb.T).squeeze(1)
        order = sims.argsort()[::-1][:q]
        picked += [cands[i] for i in order]
    # fill the rest by similarity
    remaining = [t for t in triples if t not in picked]
    if remaining and len(picked) < k:
        texts = [f"{t.s} {t.r} {t.o}" for t in remaining]
        emb = _embedder.encode(texts, normalize_embeddings=True)
        sims = (emb @ doc_emb.T).squeeze(1)
        order = sims.argsort()[::-1][: (k - len(picked))]
        picked += [remaining[i] for i in order]
    return picked[:k]

def ser_kaping_grouped_irac(trs: List[Triple]) -> str:
    if not trs: return ""
    groups = {"ISSUE":[], "RULE":[], "APPLICATION":[], "CONCLUSION":[], "FACTS":[]}
    for t in trs: groups[_irac_bucket_of(t.r)].append(t)
    lines = ["Below are facts grouped by legal reasoning sections."]
    for sec in ["ISSUE","RULE","APPLICATION","CONCLUSION","FACTS"]:
        if groups[sec]:
            lines.append(f"\n[{sec}]")
            lines += [f"({t.s}, {t.r}, {t.o})" for t in groups[sec]]
    return "\n".join(lines)



In [ ]:
def _read_first(glob_pat):
    paths = sorted(glob.glob(glob_pat, recursive=True))
    if not paths: return None, None
    p = paths[0]
    with open(p, "r", encoding="utf-8") as f:
        return f.read(), p

REPO_SUMMARY_PROMPT, REPO_PROMPT_PATH = _read_first("LegalSumAI/**/prompts/*summary*.*")
if REPO_SUMMARY_PROMPT is None:
    REPO_SUMMARY_PROMPT, REPO_PROMPT_PATH = _read_first("LegalSumAI/**/prompts/*cod*.*")
if REPO_SUMMARY_PROMPT is None:
    REPO_SUMMARY_PROMPT, REPO_PROMPT_PATH = _read_first("LegalSumAI/**/prompts/*chain_of_density*.*")

REPO_STYLE_SYSTEM_TEMPLATE = (
    "Setting: You assist users in summarizing legal materials. "
    "Your single hard requirement: match the requested length. "
    "Be clear, precise, neutral; avoid quotes/citations; output plain text."
)

def build_repo_style_user_prompt_kg_irac(length_words: int, doc_excerpt: str, triples_text: str|None,
                                         verbose: bool=False, preview_chars: int=400) -> str:
    irac_ruleset = (
        "Issue – Identify the legal question.\n"
        "Rule – State the controlling law/precedents.\n"
        "Application – Apply the rule to the facts.\n"
        "Conclusion – State the outcome.\n"
    )
    guide = f"It must have ~{length_words} words (±20%)."
    header = (
        "You are a lawyer describing the court case to the general public.\n"
        f"{guide}\nCreate a summary from the provided excerpt and knowledge graph.\n"
        f"Make sure the summary covers:\n{irac_ruleset}"
        "Use ONLY the document excerpt and the fact triples if present.\n"
        "No quotes, no citations.\n"
    )
    body = (f"\nFACT TRIPLES:\n{triples_text}\n" if triples_text else "") + \
           f"\nDOCUMENT EXCERPT:\n{doc_excerpt}\n"
    return header + body

def build_repo_style_user_prompt_kg(length_words: int, doc_excerpt: str, triples_text: str|None,
                                    verbose: bool=False, preview_chars: int=400) -> str:
    guide = f"It must have ~{length_words} words (±20%)."
    header = (
        "You are a lawyer describing the court case to the general public.\n"
        f"{guide}\nCreate a summary from the provided excerpt.\n"
        "Use ONLY the document excerpt.\n"
        "No quotes, no citations.\n"
    )
    body = f"\nDOCUMENT EXCERPT:\n{doc_excerpt}\n"
    return header + body

def _chat(system_txt: str, user_txt: str) -> str:
    # FIX: use OPENAI_CHAT_MODEL (was OPENAI_MODEL)
    resp = client.chat.completions.create(
        model=OPENAI_CHAT_MODEL,
        messages=[{"role":"system","content":system_txt},
                  {"role":"user","content":user_txt}],
        temperature=TEMPERATURE
    )
    return resp.choices[0].message.content.strip()



In [ ]:
def compute_readability(text):
    try:
        return dict(fkgl=_ts.flesch_kincaid_grade(text), fre=_ts.flesch_reading_ease(text))
    except Exception:
        return {}

def sbert_doc_sim(hyp, doc):
    E = _embedder.encode([hyp, doc], normalize_embeddings=True)
    return float((E[0] * E[1]).sum())

def _graph_from_triples(trs):
    G = nx.DiGraph()
    for t in trs:
        G.add_edge(t.s, t.o, label=t.r)
    return G

def _nx_to_svg_data_uri(G):
    from networkx.drawing.nx_pydot import to_pydot
    svg = to_pydot(G).create_svg()
    return "data:image/svg+xml;base64," + base64.b64encode(svg).decode()




In [ ]:
import gradio as gr
import tempfile, base64

def run_pipeline(doc_text: str):
    if not doc_text or not isinstance(doc_text, str):
        raise gr.Error("Please paste a legal document.")
    doc = doc_text.strip()[:MAX_DOC_CHARS]

    # 1) KGs
    kg_plain = build_plain_kg(doc)
    kg_irac  = build_irac_kg(doc)

    # 2) IRAC-Quota Top-K + Kaping-IRAC
    triples_topk = select_topk_with_irac_quota(kg_irac, doc, k=TOPK_TRIPLES)
    triples_block = ser_kaping_grouped_irac(triples_topk)

    # 3) Prompts (repo-style)
    best_prompt = build_repo_style_user_prompt_kg_irac(LENGTH_WORDS, doc, triples_block)
    base_prompt = build_repo_style_user_prompt_kg(LENGTH_WORDS, doc, None)

    # 4) Summaries
    best_summary = _chat(REPO_STYLE_SYSTEM_TEMPLATE, best_prompt)
    baseline_summary = _chat(REPO_STYLE_SYSTEM_TEMPLATE, base_prompt)

    # 5) Graph images (data URIs)
    return {
        "Best Summary": best_summary,
        "Zero-shot Baseline": baseline_summary,
        "Triples (IRAC-Quota)": [f"({t.s}, {t.r}, {t.o})" for t in triples_topk],
        "Kaping-IRAC Block": triples_block,
        "IRAC KG": _nx_to_svg_data_uri(_graph_from_triples(kg_irac)),
        "Plain KG": _nx_to_svg_data_uri(_graph_from_triples(kg_plain)),
    }

def _save_data_uri(uri: str) -> str:
    head, b64 = uri.split(",", 1)
    ext = ".svg" if "svg+xml" in head else ".png"
    fp = tempfile.NamedTemporaryFile(delete=False, suffix=ext)
    fp.write(base64.b64decode(b64)); fp.flush(); fp.close()
    return fp.name

# Prefill textbox with one of the downloaded MultiLexSum docs
try:
    demo_df = pd.read_csv("sample_multilexsum_docs.csv")
    example_doc = str(demo_df.loc[0, "document_text"])[:2000]
except Exception:
    example_doc = ""

with gr.Blocks(title="RuleSum — Best Structured vs Zero-shot") as demo:
    gr.Markdown("#### Paste a legal document (or use the prefilled sample) and click **Run**.")
    doc_box = gr.Textbox(lines=12, label="Document", value=example_doc)
    run_btn = gr.Button("Run", variant="primary")

    with gr.Row():
        with gr.Column():
            best_box = gr.Textbox(label="Best Structured Summary (IRAC KG + Quota + Kaping-IRAC)", lines=8)
        with gr.Column():
            base_box = gr.Textbox(label="Zero-shot Baseline Summary", lines=8)

    triples = gr.Dataframe(headers=["Triples (IRAC-Quota)"], row_count=5)
    kaping_box = gr.Textbox(label="Kaping-IRAC Serialization Block", lines=8)

    gr.Markdown("### Knowledge Graphs")
    with gr.Tabs():
        with gr.Tab("IRAC KG"):
            irac_img = gr.Image(label="IRAC Knowledge Graph")
        with gr.Tab("Plain KG"):
            plain_img = gr.Image(label="Plain Knowledge Graph")

    def _run(doc_text):
        out = run_pipeline(doc_text)
        return (
            out["Best Summary"],
            out["Zero-shot Baseline"],
            [[t] for t in out["Triples (IRAC-Quota)"]],
            out["Kaping-IRAC Block"],
            _save_data_uri(out["IRAC KG"]),
            _save_data_uri(out["Plain KG"]),
        )

    run_btn.click(
        _run,
        inputs=[doc_box],
        outputs=[best_box, base_box, triples, kaping_box, irac_img, plain_img]
    )

demo.launch()
